In [1]:
# Εργαλείο για τη διαχείριση διαδρομών και αρχείων
from pathlib import Path

# Βιβλιοθήκες επεξεργασίας και ανάλυσης δεδομένων
import numpy as np
import pandas as pd

# Βιβλιοθήκες δημιουργίας γραφημάτων
import matplotlib.pyplot as plt
import seaborn as sns

# Βασικές ρυθμίσεις εμφάνισης
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

print("Οι βιβλιοθήκες φορτώθηκαν επιτυχώς.")

Οι βιβλιοθήκες φορτώθηκαν επιτυχώς.


# Μέρος 1: Ανάλυση ωριαίας ενεργειακής κατανάλωσης PJME

## 1. Φόρτωση και αρχικός έλεγχος του dataset

Σε αυτή την ενότητα φορτώνουμε το dataset `PJME_hourly.csv` και πραγματοποιούμε έναν πρώτο έλεγχο της δομής και του περιεχομένου του.

In [2]:
# Ορίζουμε τη διαδρομή του dataset
data_path = Path("../Datasets/PJME_hourly.csv")

# Ελέγχουμε ότι το αρχείο υπάρχει
if not data_path.exists():
    raise FileNotFoundError("Το αρχείο PJME_hourly.csv δεν βρέθηκε.")

# Φορτώνουμε το dataset
df = pd.read_csv(data_path)

print("Το dataset φορτώθηκε επιτυχώς.")
print(f"Αριθμός γραμμών: {df.shape[0]:,}")
print(f"Αριθμός στηλών: {df.shape[1]}")

# Εμφανίζουμε τις πρώτες πέντε γραμμές
df.head()

Το dataset φορτώθηκε επιτυχώς.
Αριθμός γραμμών: 145,366
Αριθμός στηλών: 2


,Datetime,PJME_MW
0,2002-12-31 01:00:00,26498.0
1,2002-12-31 02:00:00,25147.0
2,2002-12-31 03:00:00,24574.0
3,2002-12-31 04:00:00,24393.0
4,2002-12-31 05:00:00,24860.0


## 2. Αρχικός έλεγχος των δεδομένων

Ελέγχουμε τις στήλες, τους τύπους δεδομένων, τις ελλιπείς τιμές και τις διπλές εγγραφές.

In [3]:
print("Στήλες του dataset:")
print(df.columns.tolist())

print("\nΤύποι δεδομένων:")
print(df.dtypes)

print("\nΕλλιπείς τιμές ανά στήλη:")
print(df.isna().sum())

print("\nΠλήρως διπλές γραμμές:")
print(df.duplicated().sum())

Στήλες του dataset:
['Datetime', 'PJME_MW']

Τύποι δεδομένων:
Datetime        str
PJME_MW     float64
dtype: object

Ελλιπείς τιμές ανά στήλη:
Datetime    0
PJME_MW     0
dtype: int64

Πλήρως διπλές γραμμές:
0


In [4]:
# Μετατρέπουμε τη στήλη Datetime σε μορφή ημερομηνίας
df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce")

# Ταξινομούμε τις γραμμές χρονολογικά
df = df.sort_values("Datetime").reset_index(drop=True)

# Εμφανίζουμε τις ημερομηνίες σε μορφή ημέρα/μήνας/έτος
first_date = df["Datetime"].min().strftime("%d/%m/%Y")
last_date = df["Datetime"].max().strftime("%d/%m/%Y")

print("Πρώτη ημερομηνία:", first_date)
print("Τελευταία ημερομηνία:", last_date)
print("Μη έγκυρες ημερομηνίες:", df["Datetime"].isna().sum())

Πρώτη ημερομηνία: 01/01/2002
Τελευταία ημερομηνία: 03/08/2018
Μη έγκυρες ημερομηνίες: 0


In [5]:
print("Βασικά στατιστικά της ενεργειακής κατανάλωσης:")

df["PJME_MW"].describe()

Βασικά στατιστικά της ενεργειακής κατανάλωσης:


count    145366.000000
mean      32080.222831
std        6464.012166
min       14544.000000
25%       27573.000000
50%       31421.000000
75%       35650.000000
max       62009.000000
Name: PJME_MW, dtype: float64

## 3. Έλεγχος διπλών χρονικών στιγμών και χρονικών κενών

Επειδή το dataset περιέχει ωριαίες μετρήσεις, ελέγχουμε αν υπάρχουν ημερομηνίες που εμφανίζονται περισσότερες από μία φορές ή χρονικά διαστήματα μεγαλύτερα από μία ώρα.

In [6]:
# Ελέγχουμε πόσες χρονικές στιγμές εμφανίζονται ξανά
duplicate_times = df["Datetime"].duplicated().sum()

print("Διπλές χρονικές στιγμές:", duplicate_times)

# Εμφανίζουμε όλες τις σχετικές γραμμές
duplicates = df[df["Datetime"].duplicated(keep=False)]

duplicates

Διπλές χρονικές στιγμές: 4


,Datetime,PJME_MW
112487,2014-11-02 02:00:00,23755.0
112488,2014-11-02 02:00:00,22935.0
121223,2015-11-01 02:00:00,21567.0
121224,2015-11-01 02:00:00,21171.0
130127,2016-11-06 02:00:00,20795.0
130128,2016-11-06 02:00:00,21692.0
138863,2017-11-05 02:00:00,21236.0
138864,2017-11-05 02:00:00,20666.0


In [7]:
# Υπολογίζουμε τη χρονική διαφορά ανάμεσα σε διαδοχικές μετρήσεις
time_difference = df["Datetime"].diff()

# Εντοπίζουμε διαστήματα μεγαλύτερα από μία ώρα
gaps = df[time_difference > pd.Timedelta(hours=1)].copy()

print("Χρονικά κενά μεγαλύτερα από μία ώρα:", len(gaps))

gaps.head()

Χρονικά κενά μεγαλύτερα από μία ώρα: 30


,Datetime,PJME_MW
2306,2002-04-07 04:00:00,24487.0
7176,2002-10-27 03:00:00,20605.0
11040,2003-04-06 04:00:00,22902.0
15910,2003-10-26 03:00:00,20641.0
19774,2004-04-04 04:00:00,23208.0


## 4. Καθαρισμός της χρονικής σειράς

Οι διπλές χρονικές στιγμές συνδυάζονται χρησιμοποιώντας τη μέση ενεργειακή κατανάλωση. Στη συνέχεια δημιουργείται μια συνεχόμενη ωριαία χρονική σειρά και οι ώρες που λείπουν συμπληρώνονται με γραμμική παρεμβολή.

Με αυτόν τον τρόπο διατηρείται μία μέτρηση ανά ώρα, κάτι που είναι απαραίτητο για τις επόμενες χρονικές αναλύσεις.

In [8]:
# Κρατάμε αντίγραφο των αρχικών δεδομένων
df_original = df.copy()

# Συνδυάζουμε τις διπλές ώρες χρησιμοποιώντας τον μέσο όρο
df = (
    df.groupby("Datetime", as_index=False)["PJME_MW"]
    .mean()
    .sort_values("Datetime")
)

print("Διπλές χρονικές στιγμές μετά τη διόρθωση:")
print(df["Datetime"].duplicated().sum())

Διπλές χρονικές στιγμές μετά τη διόρθωση:
0


In [9]:
# Δημιουργούμε μία πλήρη ωριαία χρονική σειρά
full_time_range = pd.date_range(
    start=df["Datetime"].min(),
    end=df["Datetime"].max(),
    freq="h"
)

# Προσθέτουμε τις ώρες που λείπουν
df = df.set_index("Datetime").reindex(full_time_range)
df.index.name = "Datetime"

missing_hours = df["PJME_MW"].isna().sum()
print("Ώρες που χρειάζονται συμπλήρωση:", missing_hours)

# Συμπληρώνουμε τις τιμές με γραμμική παρεμβολή
df["PJME_MW"] = df["PJME_MW"].interpolate(method="time")

# Επαναφέρουμε την ημερομηνία ως κανονική στήλη
df = df.reset_index()

print("Ελλιπείς τιμές μετά τον καθαρισμό:")
print(df.isna().sum())

Ώρες που χρειάζονται συμπλήρωση: 30
Ελλιπείς τιμές μετά τον καθαρισμό:
Datetime    0
PJME_MW     0
dtype: int64


In [10]:
# Κρατάμε την πλήρη ωριαία σειρά για πιθανή μεταγενέστερη χρήση
df_continuous = df.copy()

# Επαναφέρουμε τις αρχικές γραμμές του dataset
df = df_original.copy()

print("Γραμμές αρχικού dataset:", len(df))
print("Γραμμές συνεχόμενης χρονικής σειράς:", len(df_continuous))

Γραμμές αρχικού dataset: 145366
Γραμμές συνεχόμενης χρονικής σειράς: 145392


## 5. Δημιουργία συνθετικών κατηγοριών πελατών

Το αρχικό dataset δεν περιέχει πραγματικές κατηγορίες πελατών. Για τις ανάγκες της εργασίας δημιουργείται η μεταβλητή `Customer_Segment` με βάση το student ID.

Οι κατηγορίες `Commercial`, `Residential` και `Industrial` είναι συνθετικές και δεν αντιπροσωπεύουν πραγματικούς πελάτες του αρχικού dataset.

In [15]:
def allocate_customer_segments(df, student_id):
    """
    Δημιουργεί τρεις συνθετικές κατηγορίες πελατών.
    Η κατανομή τους εξαρτάται από το student ID.
    """

    # Υπολογίζουμε μια μικρή μεταβολή με βάση το student ID
    drift = ((student_id % 5) - 2) * 0.05

    # Υπολογίζουμε την πιθανότητα κάθε κατηγορίας
    prob_commercial = 0.40 + drift
    prob_residential = 0.35 - drift
    prob_industrial = 1.0 - (
        prob_commercial + prob_residential
    )

    # Αποθηκεύουμε τις πιθανότητες με τη σειρά των κατηγοριών
    weights = [
        prob_commercial,
        prob_residential,
        prob_industrial
    ]

    # Ορίζουμε τις τρεις συνθετικές κατηγορίες πελατών
    categories = [
        'Commercial',
        'Residential',
        'Industrial'
    ]

    # Χρησιμοποιούμε το student ID ώστε τα αποτελέσματα
    # να είναι ίδια κάθε φορά που εκτελείται ο κώδικας
    np.random.seed(student_id)

    # Αντιστοιχίζουμε κάθε γραμμή σε μία κατηγορία πελάτη
    df['Customer_Segment'] = np.random.choice(
        categories,
        size=len(df),
        p=weights
    )

    # Εμφανίζουμε την ποσοστιαία κατανομή των κατηγοριών
    print(
        f"\nΚατανομή κατηγοριών πελατών "
        f"για το ID {student_id}:"
    )

    print(
        df['Customer_Segment']
        .value_counts(normalize=True)
        .round(3) * 100
    )

    return df

In [16]:
# Ορίζουμε το αριθμητικό μέρος του student ID
student_id = 29901

# Εφαρμόζουμε τη συνάρτηση στο dataset
df = allocate_customer_segments(df, student_id)

# Εμφανίζουμε τις πρώτες γραμμές με τη νέα στήλη
df.head()


Κατανομή κατηγοριών πελατών για το ID 29901:
Customer_Segment
Residential    40.0
Commercial     34.9
Industrial     25.1
Name: proportion, dtype: float64


,Datetime,PJME_MW,Customer_Segment
0,2002-01-01 01:00:00,30393.0,Residential
1,2002-01-01 02:00:00,29265.0,Residential
2,2002-01-01 03:00:00,28357.0,Commercial
3,2002-01-01 04:00:00,27899.0,Commercial
4,2002-01-01 05:00:00,28057.0,Industrial


## 6. Δημιουργία χρονικών χαρακτηριστικών

Από τη στήλη `Datetime` δημιουργούμε βασικά χρονικά χαρακτηριστικά που θα χρησιμοποιηθούν στις επόμενες αναλύσεις.

Επιπλέον, δημιουργούμε τα κυκλικά χαρακτηριστικά `Month_sin` και `Month_cos` σύμφωνα με τους μαθηματικούς τύπους του handbook. Τα χαρακτηριστικά αυτά αποτυπώνουν την κυκλική φύση των μηνών, όπου ο Δεκέμβριος βρίσκεται χρονικά κοντά στον Ιανουάριο.

In [19]:
# Παίρνουμε τον μήνα και την ώρα από τη στήλη Datetime
df['t_month'] = df['Datetime'].dt.month
df['t_hour'] = df['Datetime'].dt.hour

# Εφαρμόζουμε τους τύπους του handbook
df['Month_sin'] = np.sin(
    2 * np.pi * df['t_month'] / 12
) * 3

df['Month_cos'] = np.cos(
    2 * np.pi * df['t_month'] / 12
) * 2

# Ελέγχουμε τα νέα χαρακτηριστικά
df[
    ['Datetime', 't_month', 't_hour', 'Month_sin', 'Month_cos']
].head()

,Datetime,t_month,t_hour,Month_sin,Month_cos
0,2002-01-01 01:00:00,1,1,1.5,1.732051
1,2002-01-01 02:00:00,1,2,1.5,1.732051
2,2002-01-01 03:00:00,1,3,1.5,1.732051
3,2002-01-01 04:00:00,1,4,1.5,1.732051
4,2002-01-01 05:00:00,1,5,1.5,1.732051


In [20]:
# Υπολογίζουμε τις παραμέτρους από το student ID
alpha_s = 15 + (student_id % 7) - 3
beta_s = 12 + (student_id % 5) * 0.5
gamma = 5

phi_s1 = 5 + (student_id % 4)
phi_s2 = 6 + (student_id % 4)

print("alpha_s:", alpha_s)
print("beta_s:", beta_s)
print("gamma:", gamma)
print("phi_s1:", phi_s1)
print("phi_s2:", phi_s2)

alpha_s: 16
beta_s: 12.5
gamma: 5
phi_s1: 6
phi_s2: 7


### Παραδοχές για τη συνθετική θερμοκρασία

Ο τύπος του handbook παρουσιάζει ορισμένες ασυνέπειες. Για την εφαρμογή του χρησιμοποιούνται οι ακόλουθες παραδοχές:

- Το `α_s` χρησιμοποιείται ως σταθερή βασική μέση θερμοκρασία.
- Ο εποχικός όρος υπολογίζεται από τον μήνα.
- Ο ημερήσιος όρος υπολογίζεται από την ώρα.
- Το `γ` χρησιμοποιείται ως το πλάτος της ημερήσιας μεταβολής.
- Ο τυχαίος όρος θεωρείται ότι ακολουθεί κανονική κατανομή `N(0,1)`.
- Το αριθμητικό student ID χρησιμοποιείται ως seed, ώστε τα αποτελέσματα να είναι επαναλήψιμα.

Η μεταβλητή είναι συνθετική και δεν αποτελεί πραγματική μετεωρολογική μέτρηση.

In [27]:
# Χρησιμοποιούμε το student ID ως seed,
# ώστε το σφάλμα να είναι ίδιο σε κάθε εκτέλεση
np.random.seed(student_id)

# Δημιουργούμε το τυχαίο σφάλμα με την παραδοχή N(0, 1)
e = np.random.normal(
    loc=0,
    scale=1,
    size=len(df)
)

# Υπολογίζουμε πρώτα τη βασική συνθετική θερμοκρασία
df['T_base'] = (
    alpha_s
    + beta_s
    * np.sin(
        2 * np.pi * (df['t_month'] - phi_s1) / 12
    )
    + gamma
    * np.sin(
        2 * np.pi * (df['t_hour'] - phi_s2) / 24
    )
)

# Προσθέτουμε το τυχαίο σφάλμα
df['Synthetic_Temperature'] = df['T_base'] + e

# Εμφανίζουμε τις πρώτες τιμές
df[
    [
        'Datetime',
        't_month',
        't_hour',
        'T_base',
        'Synthetic_Temperature'
    ]
].head()

,Datetime,t_month,t_hour,T_base,Synthetic_Temperature
0,2002-01-01 01:00:00,1,1,4.750000,4.908232
1,2002-01-01 02:00:00,1,2,4.920371,6.981854
2,2002-01-01 03:00:00,1,3,5.419873,5.184121
3,2002-01-01 04:00:00,1,4,6.214466,5.880744
4,2002-01-01 05:00:00,1,5,7.250000,7.939935


In [29]:
# Ελέγχουμε τις βασικές πληροφορίες των δύο νέων μεταβλητών
df[['T_base', 'Synthetic_Temperature']].describe().round(2)

,T_base,Synthetic_Temperature
count,145366.00,145366.00
mean,15.86,15.87
std,9.49,9.54
min,-1.50,-4.90
25%,7.83,7.81
50%,14.75,15.66
75%,23.67,23.92
max,33.50,36.80


In [30]:
# Ελέγχουμε αν υπάρχουν ελλιπείς τιμές
df[['T_base', 'Synthetic_Temperature']].isna().sum()

T_base                   0
Synthetic_Temperature    0
dtype: int64